# SNR-attention detection adaptation — raw vs SNR-enhanced ExDark

This notebook completes the planned detector fine-tuning comparison. YOLOv8m is fine-tuned from the same COCO weights and seed on (1) raw ExDark training images and (2) images enhanced by the selected SNR-guided Retinexformer. The official ExDark validation split selects detector checkpoints; both detectors are then evaluated on the same existing 1,200-image official test subset, once raw and once SNR-enhanced.

**Kaggle settings:** GPU T4 x2, Internet ON. Attach the output of `snr_attention_experiments.ipynb` so its selected Retinexformer checkpoint is available. The ExDark test subset is never used to train either detector.

In [ ]:
import glob, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

os.chdir('/kaggle/working')
if not Path('btp').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Crazycraftie/low-light-enhancement-btp.git', 'btp'], check=True)
os.chdir('/kaggle/working/btp')
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/caiyuanhao1998/Retinexformer.git', 'baselines/Retinexformer'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'gdown', 'lmdb', 'yacs', 'addict', 'future', 'natsort', 'yapf', 'thop', 'timm', 'ultralytics'], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before running this notebook.'
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2: this notebook launches one matched detector per GPU.'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

archives = glob.glob('/kaggle/input/**/snr_attention_state.zip', recursive=True)
if not archives:
    raise FileNotFoundError('Attach the output of snr_attention_experiments.ipynb containing snr_attention_state.zip.')
with zipfile.ZipFile(archives[0]) as zf:
    zf.extractall('/kaggle/working/btp')
print('Restored enhancement sweep/final checkpoints from', archives[0])

sweep = []
for alpha in (2, 4, 8):
    values = [json.loads(Path(f'runs/snr_attention/sweep/alpha{alpha}_seed{seed}/metrics.json').read_text())['best_val_psnr']
              for seed in (42, 43, 44)]
    sweep.append((sum(values) / len(values), alpha))
BEST_ALPHA = max(sweep, key=lambda row: (row[0], -row[1]))[1]
FINAL_RUN = Path(f'runs/snr_attention/final/snr_attention_a{BEST_ALPHA}')
MODEL = FINAL_RUN / 'best.pth'
assert (FINAL_RUN / 'metrics.json').is_file(), 'Finish the full SNR-guided training run before detection fine-tuning.'
assert MODEL.is_file(), f'Missing selected SNR-guided checkpoint: {MODEL}'
print('Selected SNR-guided model:', MODEL, '| alpha:', BEST_ALPHA)

In [ ]:
exdark = Path('data/ExDark')
if not (exdark / 'imageclasslist.txt').is_file():
    Path('/kaggle/tmp/exdark').mkdir(parents=True, exist_ok=True)
    for file_id, name in [('1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx', 'images'),
                          ('1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i', 'annotations')]:
        subprocess.run(['gdown', '-q', file_id, '-O', f'/kaggle/tmp/exdark/{name}.zip'], check=True)
        subprocess.run(['unzip', '-q', '-o', f'/kaggle/tmp/exdark/{name}.zip', '-d', '/kaggle/tmp/exdark/unpacked'], check=True)
    exdark.mkdir(parents=True, exist_ok=True)
    shutil.move('/kaggle/tmp/exdark/unpacked/ExDark', exdark / 'images')
    shutil.move('/kaggle/tmp/exdark/unpacked/ExDark_Annno', exdark / 'annotations')
    subprocess.run(['curl', '-fsSL',
                    'https://raw.githubusercontent.com/cs-chan/Exclusively-Dark-Image-Dataset/master/Groundtruth/imageclasslist.txt',
                    '-o', str(exdark / 'imageclasslist.txt')], check=True)

if not Path('data/exdark_yolo/raw/images').is_dir():
    subprocess.run([sys.executable, 'scripts/exdark_to_yolo.py'], check=True)
subprocess.run([sys.executable, 'scripts/prepare_exdark_yolo_train.py', '--prepare-raw'], check=True)
test_count = len(list(Path('data/exdark_yolo/raw/images').glob('*.png')))
assert test_count == 1200, f'Expected the fixed 1,200-image ExDark test subset, found {test_count}'
print('Prepared official ExDark train and validation splits; fixed held-out test images:', test_count)

## Enhance training, validation, and fixed test images

Only train/validation images are used for detector learning and checkpoint selection. The 1,200 test images were reserved by the earlier project evaluation.

In [ ]:
raw_train = Path('data/exdark_yolo/finetune/raw/train/images')
raw_val = Path('data/exdark_yolo/finetune/raw/val/images')
outputs = {
    'train': Path('results/snr_attention_detector/snr_train'),
    'val': Path('results/snr_attention_detector/snr_val'),
    'test': Path('results/snr_attention_exdark/ExDark'),
}
sources = {'train': raw_train, 'val': raw_val, 'test': Path('data/exdark_yolo/raw/images')}
for split, source in sources.items():
    output = outputs[split]
    expected = {p.name for p in source.glob('*.png')}
    existing = {p.name for p in output.glob('*.png')} if output.is_dir() else set()
    if existing != expected:
        subprocess.run([sys.executable, 'scripts/retinexformer_infer.py', '--repo', 'baselines/Retinexformer',
                        '--opt', 'baselines/Retinexformer/Options/RetinexFormer_LOL_v1.yml',
                        '--weights', str(MODEL), '--input', str(source), '--output', str(output),
                        '--method', f'SNR-attention-a{BEST_ALPHA}', '--device', '0', '--no-speed-log',
                        '--snr-attention', '--alpha', str(BEST_ALPHA)], check=True)
    print(split, len(expected), 'images ->', output)

subprocess.run([sys.executable, 'scripts/prepare_exdark_yolo_train.py', '--domain', 'snr_attention',
                '--train-images', str(outputs['train']), '--val-images', str(outputs['val'])], check=True)
subprocess.run([sys.executable, 'scripts/build_yolo_sets.py', '--methods', 'snr_attention_exdark'], check=True)

## Fine-tune matched detectors and compare on identical test images

Both detectors start from `yolov8m.pt`, use the same seed and training settings, and differ only in whether training pixels are raw or SNR-enhanced. Each detector is tested on both pixel domains to reveal domain adaptation as well as overall detection quality.

In [ ]:
# Download/cache identical COCO initialization before launching concurrent jobs.
subprocess.run([sys.executable, '-c', 'from ultralytics import YOLO; YOLO("yolov8m.pt")'], check=True)
jobs = [('raw', 'data/exdark_yolo/finetune/raw.yaml'),
        ('snr_attention', 'data/exdark_yolo/finetune/snr_attention.yaml')]
processes = []
for gpu, (name, data_yaml) in enumerate(jobs):
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    log = open(f'/kaggle/working/{name}_detector.log', 'w')
    cmd = [sys.executable, 'scripts/train_yolo_exdark.py', '--data', data_yaml, '--name', name,
           '--model', 'yolov8m.pt', '--epochs', '50', '--imgsz', '1024', '--batch', '4',
           '--seed', '42', '--device', '0', '--workers', '4']
    processes.append((name, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env), log))
for name, process, log in processes:
    code = process.wait()
    log.close()
    print(name, 'exit code', code)
    if code != 0:
        raise RuntimeError(Path(f'/kaggle/working/{name}_detector.log').read_text()[-8000:])

for detector in ('raw', 'snr_attention'):
    weights = f'runs/snr_attention_detector/{detector}/weights/best.pt'
    subprocess.run([sys.executable, 'scripts/yolo_eval.py', '--model', weights,
                    '--sets', 'raw', 'snr_attention_exdark', '--result-prefix', f'{detector}-trained-',
                    '--device', '0'], check=True)

with zipfile.ZipFile('/kaggle/working/snr_attention_detection_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in Path('runs/snr_attention_detector').rglob('best.pt'):
        zf.write(path, path.as_posix())
    for path in (Path('results/detection.csv'), Path('results/detection_per_class.csv')):
        if path.exists():
            zf.write(path, path.name)
    for path in Path('results/figures/detection').rglob('*.png'):
        if 'raw-trained' in str(path) or 'snr_attention-trained' in str(path):
            zf.write(path, path.as_posix())
print('Saved models and test metrics: /kaggle/working/snr_attention_detection_results.zip')